# 04 — RSSM, Latent Imagination and SSMs

The deterministic GRU stores one best guess. The RSSM stores a **distribution over possible hidden worlds**. The SSM stores a persistent state with structured decay and selective updates.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))

In [2]:
import torch
from apexsim.models.rssm import RSSMWorldModel
from apexsim.models.ssm_world_model import SSMWorldModel

history = torch.randn(3, 12, 16)
future = torch.randn(3, 5, 16)
target = torch.randn(3, 5, 5)
rssm = RSSMWorldModel(16, 5, hidden_dim=32, latent_dim=8)
ssm = SSMWorldModel(16, 5, hidden_dim=32, layers=2)
rssm_pred, kl = rssm(history, future, target)
ssm_pred = ssm(history, future)
print(rssm_pred.shape, 'KL=', float(kl))
print(ssm_pred.shape)

torch.Size([3, 5, 5]) KL= 0.7861248850822449
torch.Size([3, 5, 5])


/tmp/ipykernel_15114/99702834.py:12: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(rssm_pred.shape, 'KL=', float(kl))


## Dreamer-style RSSM mental model

```text
previous latent + action/context -> deterministic memory h_t
                                      |
                    prior p(z_t | h_t)
                                      |
observed state during training -> posterior q(z_t | h_t, observation)
                                      |
                         decoder predicts state
```

Reconstruction teaches usefulness. KL divergence teaches the prior to resemble the observation-informed posterior so imagination remains possible when future observations disappear.

## SSM mental model

A classical state-space recurrence is:

`h_t = A h_(t-1) + B x_t`, `y_t = C h_t`.

Structured SSMs make this recurrence stable and efficient for long sequences. Selective SSMs make update behaviour depend on the input, allowing the model to retain important events and forget irrelevant ones. The included cell is pedagogical rather than a full hardware-optimized Mamba implementation.

In [3]:
decay = torch.sigmoid(ssm.cells[0].logit_decay).detach().numpy()
print('Learned initial decay range:', decay.min(), decay.max())

Learned initial decay range: 0.5 0.5


## Choosing among them

- **GRU:** strongest first implementation; simple, stable and causal.
- **RSSM:** use when multiple futures or uncertainty matter; harder to optimize and evaluate.
- **SSM/Mamba:** use when histories become very long or streaming efficiency matters.
- **Transformer:** use when flexible pairwise attention provides measured value and sequence length is manageable.

Architecture is a response to a bottleneck, not a badge.